# UCS420 Cognitive Computing
## NLP using Python-II
**Topics:** Text preprocessing, stemming and lemmatization, Bag of Words, TF-IDF, text similarity, sentiment analysis, and text generation.  




## Setup — Install and import libraries
The first cell installs the additional libraries needed in Google Colab. TensorFlow is normally preinstalled in Colab, but a fallback installation is provided. NLTK corpora are downloaded before use.

In [ ]:
%pip -q install nltk scikit-learn textblob wordcloud matplotlib pandas

In [ ]:
import re
import string
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

for package in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4']:
    nltk.download(package, quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from textblob import TextBlob
from wordcloud import WordCloud

STOPWORDS = set(stopwords.words('english'))
print('Setup completed successfully.')

## Q1. Text preprocessing and word frequency
**Task:** Write an original 5–6 sentence paragraph, convert it to lowercase, remove punctuation using `re`, tokenize into sentences and words, compare `split()` with NLTK's `word_tokenize()`, remove stopwords, and display the word-frequency distribution.

In [ ]:
paragraph = (
    "Artificial intelligence is changing how students learn and solve problems. "
    "Smart tutoring systems can explain difficult concepts using personalized examples. "
    "Machine learning helps computers identify patterns in large datasets. "
    "However, students should question automated answers instead of trusting every prediction. "
    "Responsible technology combines human creativity with careful evaluation. "
    "I enjoy exploring these tools because they make experimentation more accessible."
)

# Tokenize sentences before punctuation is removed so boundaries remain clear.
sentences = sent_tokenize(paragraph)
lowercase_text = paragraph.lower()
cleaned_text = re.sub(r'[^a-z\s]', '', lowercase_text)
split_tokens = cleaned_text.split()
nltk_tokens = word_tokenize(cleaned_text)
filtered_tokens = [word for word in nltk_tokens if word not in STOPWORDS]
frequencies = Counter(filtered_tokens)

print('ORIGINAL PARAGRAPH:\n', paragraph)
print('\nLOWERCASE AND WITHOUT PUNCTUATION:\n', cleaned_text)
print('\nSENTENCE TOKENIZATION:')
for i, sentence in enumerate(sentences, 1):
    print(f'{i}. {sentence}')
print('\nPYTHON SPLIT TOKENS:\n', split_tokens)
print('\nNLTK WORD TOKENS:\n', nltk_tokens)
print('\nWORDS WITHOUT STOPWORDS:\n', filtered_tokens)
print('\nWORD FREQUENCY DISTRIBUTION:')
print(pd.DataFrame(frequencies.most_common(), columns=['Word', 'Frequency']).to_string(index=False))

In [ ]:
# Demonstrate why split() and word_tokenize() are not always equivalent.
comparison_text = "AI-powered tools can't replace a student's curiosity!"
print('Example:', comparison_text)
print('split():', comparison_text.split())
print('word_tokenize():', word_tokenize(comparison_text))

plt.figure(figsize=(11, 4))
words, counts = zip(*frequencies.most_common(15))
plt.bar(words, counts)
plt.xticks(rotation=55, ha='right')
plt.title('Top 15 Word Frequencies (Stopwords Removed)')
plt.xlabel('Word')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

**Interpretation:** `split()` separates text only at whitespace, so punctuation stays attached to words. NLTK's `word_tokenize()` recognizes many punctuation marks and contractions as distinct tokens. On text that has *already had punctuation removed*, both token lists will usually look similar. Removing common stopwords emphasizes more meaningful topic words.

## Q2. Regular expressions, stemming, and lemmatization
**Task:** Reuse Q1's paragraph, extract alphabetic words with `re.findall()`, remove stopwords, apply Porter stemming and WordNet lemmatization, and compare the results.

In [ ]:
alphabetic_words = re.findall(r'[A-Za-z]+', paragraph.lower())
content_words = [word for word in alphabetic_words if word not in STOPWORDS]
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

stemmed_words = [stemmer.stem(word) for word in content_words]
# Specify the verb part of speech to better normalize verb forms.
lemmatized_words = [lemmatizer.lemmatize(word, pos='v') for word in content_words]

comparison_df = pd.DataFrame({
    'Original word': content_words,
    'Porter stem': stemmed_words,
    'WordNet lemma (verb)': lemmatized_words
})
print('Alphabet-only words:', alphabetic_words)
print('\nAfter removing stopwords:', content_words)
print('\nSTEMMING AND LEMMATIZATION COMPARISON:')
print(comparison_df.to_string(index=False))

**Comparison:** Stemming mechanically removes or changes word endings, sometimes producing non-dictionary forms (for example, *changing* → *chang*). Lemmatization attempts to return a valid base form (for example, *changing* → *change* when treated as a verb). **Choose stemming** for fast keyword matching or approximate search, and **lemmatization** when readable word forms and linguistic meaning matter. The WordNet result also depends on the supplied part of speech.

## Q3. Bag of Words and TF-IDF feature extraction
**Task:** Write three short original texts, show a Bag of Words representation, calculate TF-IDF, and identify and interpret the three highest-scoring keywords in each text.

In [ ]:
texts = [
    'The robotics club designed a robot that sorts recyclable waste automatically.',
    'Our college library provides digital books and quiet study rooms for students.',
    'A solar energy project uses sunlight to generate clean electricity for homes.'
]

bow_vectorizer = CountVectorizer(stop_words='english')
bow_matrix = bow_vectorizer.fit_transform(texts)
bow_df = pd.DataFrame(bow_matrix.toarray(), columns=bow_vectorizer.get_feature_names_out(),
                      index=['Text 1', 'Text 2', 'Text 3'])
print('BAG OF WORDS (word counts):')
display(bow_df)

tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(texts)
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out(),
                        index=['Text 1', 'Text 2', 'Text 3'])
print('TF-IDF MATRIX:')
display(tfidf_df.round(3))

feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
print('TOP 3 KEYWORDS PER TEXT:')
for i, row in enumerate(tfidf_matrix.toarray(), start=1):
    # Stable tie-breaking in alphabetical order for equal TF-IDF scores.
    ranked = sorted(zip(feature_names, row), key=lambda x: (-x[1], x[0]))
    top3 = [(word, score) for word, score in ranked if score > 0][:3]
    print(f'Text {i}: ' + ', '.join(f'{word} ({score:.3f})' for word, score in top3))

**Interpretation:** Bag of Words records the number of times each vocabulary word appears; it does not account for how distinctive a word is. TF-IDF weighs terms by their importance in a document relative to the small collection. Here, the three texts concern **robotics/recycling**, **library/studying**, and **solar energy**. Because many terms appear in only one document, multiple TF-IDF values may tie; tied keywords are still representative of their document.

## Q4. Jaccard and cosine similarity between technologies
**Task:** Write two 4–6-line descriptions of different technologies, preprocess the texts, compute Jaccard similarity using word sets, compute cosine similarity using TF-IDF, and interpret the comparison.

In [ ]:
ai_text = """Artificial intelligence analyzes data to recognize patterns.
Machine learning models support automated predictions.
Intelligent software assists people with complex decisions.
AI applications use algorithms to improve digital services.
Data quality matters for accurate and fair systems."""

blockchain_text = """Blockchain technology stores records across distributed computers.
Digital transactions are verified through network consensus.
Cryptographic methods protect records against unauthorized changes.
Blockchain applications help track assets and improve transparency.
Secure data management matters for trustworthy digital systems."""

def preprocess_for_similarity(text):
    words = re.findall(r'[a-z]+', text.lower())
    return [word for word in words if word not in STOPWORDS]

ai_tokens = preprocess_for_similarity(ai_text)
blockchain_tokens = preprocess_for_similarity(blockchain_text)
set_ai, set_blockchain = set(ai_tokens), set(blockchain_tokens)
shared = set_ai & set_blockchain
union = set_ai | set_blockchain
jaccard_score = len(shared) / len(union) if union else 0.0

similarity_vectorizer = TfidfVectorizer(stop_words='english')
similarity_matrix = similarity_vectorizer.fit_transform([ai_text, blockchain_text])
cosine_score = cosine_similarity(similarity_matrix[0], similarity_matrix[1])[0, 0]

print('AI tokens:', ai_tokens)
print('Blockchain tokens:', blockchain_tokens)
print('Shared unique words:', sorted(shared))
print('Number of shared words:', len(shared))
print('Number of distinct words in union:', len(union))
print(f'Jaccard similarity: {jaccard_score:.4f}')
print(f'Cosine similarity (TF-IDF): {cosine_score:.4f}')

**Analysis:** Jaccard measures the proportion of *shared unique words* and ignores frequency. TF-IDF cosine similarity measures overlap in *weighted term usage*, reducing the influence of words that occur across both documents. In these descriptions, both scores are expected to be relatively low because AI and blockchain use different technical vocabularies, despite both involving digital systems and data. **TF-IDF cosine similarity gives the richer insight here**, because it considers term importance rather than only membership in a set. Neither method understands synonyms or deep semantic similarity.

## Q5. Sentiment analysis and positive-review word cloud
**Task:** Write short product/service reviews, compute **polarity** and **subjectivity** with TextBlob, classify each as Positive, Negative, or Neutral, and create a word cloud from positive reviews. Multiple reviews make the requested classification and positive-review word cloud meaningful.

In [ ]:
reviews = [
    'The wireless headphones have excellent sound quality and a comfortable fit.',
    'This app is easy to use and the customer support is wonderful.',
    'The battery life is terrible and the connection is unreliable.',
    'Delivery was slow and the product packaging was disappointing.',
    'The laptop arrived in a cardboard box with a charging cable.',
    'The camera is amazing and the photos look beautiful in daylight.'
]

def sentiment_label(polarity, threshold=0.05):
    if polarity > threshold:
        return 'Positive'
    if polarity < -threshold:
        return 'Negative'
    return 'Neutral'

sentiment_rows = []
for review in reviews:
    result = TextBlob(review).sentiment
    sentiment_rows.append({
        'Review': review,
        'Polarity': round(result.polarity, 3),
        'Subjectivity': round(result.subjectivity, 3),
        'Sentiment': sentiment_label(result.polarity)
    })

sentiment_df = pd.DataFrame(sentiment_rows)
display(sentiment_df)
print('Sentiment counts:')
print(sentiment_df['Sentiment'].value_counts().to_string())

In [ ]:
positive_reviews = sentiment_df.loc[sentiment_df['Sentiment'] == 'Positive', 'Review'].tolist()
positive_text = ' '.join(positive_reviews)
if positive_text:
    cloud = WordCloud(width=1000, height=500, background_color='white',
                      stopwords=STOPWORDS, collocations=False, random_state=42).generate(positive_text)
    plt.figure(figsize=(12, 5))
    plt.imshow(cloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('Word Cloud from Positive Reviews')
    plt.show()
else:
    print('No positive reviews were found; the word cloud cannot be generated.')

**Interpretation:** Polarity ranges from **−1** (negative) to **+1** (positive); subjectivity ranges from **0** (more objective) to **1** (more subjective). Here, the classification uses a small neutral band from −0.05 to +0.05. The word cloud visualizes frequently used terms in positive reviews only. TextBlob is lexicon-based, so sarcasm, context, and domain-specific vocabulary may lead to imperfect classifications.

## Q6. Next-word text generation using an LSTM
**Task:** Use an original paragraph of approximately 100 words as training data, tokenize it using Keras `Tokenizer()`, construct input sequences, train a simple LSTM, and generate **three new lines** from a supplied seed phrase.

**Note:** This is a small educational model trained on a tiny dataset, so it will mostly reproduce patterns from the training paragraph rather than writing novel, fluent text. Results can differ between runs.

In [ ]:
training_text = (
    "Technology encourages curious people to explore new ideas and solve everyday problems. "
    "In a small classroom students discuss how intelligent systems can support learning and creativity. "
    "They collect examples organize information and experiment with simple computer models. "
    "Each project begins with a clear question and grows through careful observation. "
    "Students share their results compare different approaches and learn from unexpected mistakes. "
    "Responsible innovation requires patience fairness and respect for human needs. "
    "When technology is designed thoughtfully it can make education more engaging and accessible. "
    "The best discoveries often come from teamwork curiosity and a willingness to keep improving. "
    "New tools create opportunities for students to imagine better solutions."
)
print('Training paragraph word count:', len(training_text.split()))
print(training_text)

In [ ]:
# Colab usually includes TensorFlow. Install it only if unavailable.
try:
    import tensorflow as tf
except ImportError:
    %pip -q install tensorflow
    import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

np.random.seed(42)
tf.random.set_seed(42)

tokenizer = Tokenizer(oov_token='<OOV>')
tokenizer.fit_on_texts([training_text])
total_words = len(tokenizer.word_index) + 1

sequences = []
for sentence in sent_tokenize(training_text):
    token_list = tokenizer.texts_to_sequences([sentence])[0]
    for end in range(2, len(token_list) + 1):
        sequences.append(token_list[:end])

max_sequence_length = max(len(seq) for seq in sequences)
padded_sequences = pad_sequences(sequences, maxlen=max_sequence_length, padding='pre')
X = padded_sequences[:, :-1]
y = padded_sequences[:, -1]
y_categorical = to_categorical(y, num_classes=total_words)

print('Vocabulary size:', total_words)
print('Number of training sequences:', len(sequences))
print('Input shape:', X.shape)
print('Target shape:', y_categorical.shape)
print('Example input sequence:', X[0])
print('Example target word ID:', y[0])

In [ ]:
model = Sequential([
    tf.keras.Input(shape=(max_sequence_length - 1,)),
    Embedding(input_dim=total_words, output_dim=32),
    LSTM(64),
    Dense(total_words, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Increase epochs if you want better memorization at the cost of extra training time.
history = model.fit(X, y_categorical, epochs=60, batch_size=16, verbose=0)

plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Training loss')
plt.title('LSTM Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Categorical cross-entropy')
plt.legend()
plt.tight_layout()
plt.show()
print('Final training accuracy:', round(history.history['accuracy'][-1], 3))

In [ ]:
def generate_text(seed_text, next_words=12):
    generated = seed_text
    # Exclude padding (index 0) and the unknown-word placeholder from choices.
    oov_index = tokenizer.word_index.get('<OOV>')
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([generated])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_length - 1, padding='pre')
        probabilities = model.predict(token_list, verbose=0)[0].copy()
        probabilities[0] = 0
        if oov_index is not None:
            probabilities[oov_index] = 0
        predicted_index = int(np.argmax(probabilities))
        next_word = tokenizer.index_word.get(predicted_index)
        if not next_word:
            break
        generated += ' ' + next_word
    return generated

seed_phrases = ['technology encourages', 'students share', 'responsible innovation']
print('THREE GENERATED LINES:')
for i, seed in enumerate(seed_phrases, 1):
    print(f'{i}. {generate_text(seed, next_words=12)}')

**Explanation and conclusion:** The Tokenizer maps words to integer IDs. Each sentence is expanded into progressively longer sequences, where all but the last word form the input and the last word is the target. Padding makes inputs the same length; an Embedding layer learns compact word representations; an LSTM processes word order; and a Dense softmax predicts the next word. Generating multiple words repeatedly applies that prediction. Since the corpus contains only ~100 words, generated text may be repetitive; a larger dataset and more training would improve variety and fluency.